# Model Upgrade: Evaluation-Driven Migration and Prompt Optimization

This notebook runs an **evaluation-driven model migration**. It answers two
questions a model upgrade actually hinges on:

1. **Does the upgrade hold or improve quality?** Score the current prompt on both the
   source and target models, on our own gold data, and compare accuracy, latency, and cost.
2. **Is the old prompt even right for the new model?** Re-optimize the prompt *for the
   target model* with a notebook-driven hill climb, and report the prompt that scores highest.

**Workload** is the travel-booking chatbot from the [Chatbot](../Chatbot/) module, scored
the same way (session-level goal completion + binary per-criterion rubrics).

**Architecture** is AWS-native and AgentCore-native: agents run on **AgentCore Runtime**
behind a **single AgentCore Gateway** (inference targets routing to Bedrock for the target
model and OpenAI first-party for the source model), traces land in **Observability**, and
**AgentCore custom evaluators** do the scoring. The hill-climbing control loop stays here in
the notebook so you can see *why* each candidate moved the score.

> **Runs offline by default.** `model_config.RUN.use_committed_run = True` reads a committed
> run so the whole story renders with **no inference spend**. Flip it off (and turn the
> fan-out knobs up in `model_config.py`) to run live on your own data — that is the
> intended way to use this in anger.

## The Loop

```
 seed  -> start from the current production prompt (scored on both models = baseline)
 mutate -> an LLM proposes K candidate prompts, informed by the judge's reasoning on
           where the current best prompt loses points (hill climbing, not random)
 fan out -> run all K candidates in parallel across AgentCore Runtime sessions, through
            the Gateway, against the TARGET model, over a slice of the gold scenarios
 score  -> AgentCore custom evaluators: goal-completion + per-criterion pass/fail + reasoning,
           plus latency and cost per candidate
 select -> keep the highest-scoring candidate as the new incumbent
 check  -> accuracy gain below threshold? stop. else mutate again.
 report -> the prompt that scores highest on the target model, at this accuracy/latency/cost
```

## Section 1 — Environment & Access

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import sys, os, json
sys.path.insert(0, '.')
import boto3
import model_config as mc

print('Region         :', mc.AWS_REGION)
print('Source model   :', mc.SOURCE_MODEL.label, '->', mc.SOURCE_MODEL.gateway_model)
print('Target model   :', mc.TARGET_MODEL.label, '->', mc.TARGET_MODEL.gateway_model)
print('Judge (fixed)  :', mc.JUDGE_MODEL.label)
print('Optimizer      :', mc.OPTIMIZER_MODEL.label)
print('Offline mode   :', mc.RUN.use_committed_run, '(reads committed run; no spend)')
print('Fan-out/gen    :', mc.RUN.sessions_per_generation, 'sessions',
      f'({mc.RUN.candidates_per_generation} candidates x {mc.RUN.sessions_per_candidate})')

In [ ]:
# Credentials + target model access (skipped automatically in offline mode).
if not mc.RUN.use_committed_run:
    sts = boto3.client('sts')
    print('Account:', sts.get_caller_identity()['Account'])
    br = boto3.client('bedrock-runtime', region_name=mc.AWS_REGION)
    r = br.converse(modelId=mc.TARGET_MODEL.model_id,
                    messages=[{'role':'user','content':[{'text':'Say hello in one word.'}]}],
                    inferenceConfig={'maxTokens': 10})
    print('Target OK:', r['output']['message']['content'][0]['text'])
    assert os.environ.get('OPENAI_API_KEY'), 'OPENAI_API_KEY must be set for the source model'
    print('OpenAI key present for source model (value never printed).')
else:
    print('Offline mode — skipping live access checks.')

## Section 2 — The Single Gateway Seam

An AgentCore Gateway supports three target categories — MCP, HTTP, and **inference**. An
**inference target** routes LLM traffic to one or more providers through a unified endpoint,
selecting the destination by the **`model` field** in the request. We create **one gateway**
with two inference targets:

| Target | Backend | Models | Outbound auth |
|---|---|---|---|
| `bedrock` | `bedrock-runtime` connector (native InvokeModel) | GPT-5.6 Luna + GPT-6 Luna (default) | Gateway IAM role |
| `openai` *(example)* | `https://api.openai.com` | GPT-4.1 | API key (token vault) |
| `azure` *(example)* | your Azure resource endpoint | your deployment | API key (token vault) |

The default live path uses only the `bedrock` target (the `bedrock-runtime` connector — the
deprecation-safe native-InvokeModel path, NOT `bedrock-mantle`). The agent talks to **one
endpoint** (`.../inference/v1/chat/completions`) with a single OpenAI-compatible client and
picks the backend purely by the `model` string (e.g. `bedrock/us.openai.gpt-6-luna`) — so
switching the model under test never touches agent code. Provision with
`python gateway/setup_gateway.py create` (add `--with-openai` for the OpenAI example target).

In [ ]:
# Build ONE OpenAI-compatible client against the Gateway; route by the `model` field.
# (Offline mode doesn't call out — this shows the exact seam you'd use live.)
def gateway_model_client():
    from openai import OpenAI
    return OpenAI(base_url=mc.gateway_inference_base_url(),
                  api_key=os.environ['AGENTCORE_GATEWAY_TOKEN'])

def route(model_spec):
    '''Return the `model` string the agent sends; the Gateway routes on it.'''
    return model_spec.gateway_model

print('source route:', route(mc.SOURCE_MODEL))
print('target route:', route(mc.TARGET_MODEL))
if not mc.RUN.use_committed_run:
    _client = gateway_model_client()
    print('Gateway client base_url:', _client.base_url)

## Section 3 — The Workload Under Test

The travel-booking agent (six tools, mock data, confirm-before-book) is vendored from the
Chatbot module into `agent/travel_agent.py`. Its model call is routed through the Gateway
seam above. For a live run it is packaged once for **AgentCore Runtime**; fan-out is achieved
by invoking that one runtime with many concurrent **session IDs** (each session gets its own
isolated microVM) — not by deploying many instances.

In [ ]:
from agent.travel_agent import SYSTEM_PROMPT as SEED_PROMPT, ALL_TOOLS
print('Tools:', [t.__name__ for t in ALL_TOOLS])
print('\nSeed (current production) system prompt:\n')
print(SEED_PROMPT)

In [ ]:
# Gold-standard scenarios (from the Chatbot module), each with a binary success assertion.
scenarios = [json.loads(l) for l in open('data/gold_scenarios.jsonl')]
print(len(scenarios), 'gold scenarios:')
for s in scenarios:
    print(' -', s['name'], '| failure modes:', s['metadata'].get('failure_modes'))

## Section 4 — Scoring (reused from the Chatbot module)

Two AgentCore custom evaluators, judge held **fixed** (`mc.JUDGE_MODEL`) so the score compares
the two agent models, not two judges:

- **`goal_success`** — session-level binary goal completion vs each scenario's
  `expected_assertion` → the **headline accuracy** the hill climb optimizes.
- **binary rubric checks** (F1 no invented flight, F6 no off-topic, F7 valid dates) → per-
  criterion pass/fail, each returning judge **reasoning** that steers the next mutation.

Alongside accuracy, each candidate reports **latency** (per multi-turn session) and **cost**
(tokens x price), so the final call covers all three axes.

In [ ]:
import evaluators as ev
print('Session evaluator : goal_success (assertion mode)')
print('Rubric evaluators :', list(ev.BINARY_RUBRICS))
# In a live run these register as AgentCore custom evaluators and score the Runtime sessions'
# traces from Observability. In offline mode we read their verdicts from the committed run.
for name, rubric in ev.BINARY_RUBRICS.items():
    print(f'\n[{name}]\n{rubric}')

## Section 5 — Baseline: Same Prompt, Both Models

Run the **current production prompt** on the source (GPT-5.6 Luna) and target (GPT-6 Luna) models
over the gold slice, scored identically. This alone answers *does the raw swap hold quality?*

In [ ]:
import pandas as pd
from committed_run_loader import load_committed_baseline

if mc.RUN.use_committed_run:
    base = load_committed_baseline()
    src, tgt = base['source'], base['target_unoptimized']
else:
    raise NotImplementedError('Live baseline runs via the Runtime fan-out helper; '
                              'see scripts/run_optimization.py baseline.')

pd.DataFrame([
    {'model': mc.SOURCE_MODEL.label, 'accuracy': src.accuracy,
     'latency_p50_s': src.latency_p50_s, 'cost_usd_per_session': src.cost_usd_per_session},
    {'model': mc.TARGET_MODEL.label + ' (unoptimized)', 'accuracy': tgt.accuracy,
     'latency_p50_s': tgt.latency_p50_s, 'cost_usd_per_session': tgt.cost_usd_per_session},
])

On the **unchanged** prompt the target typically *matches* the source rather than clearly
beating it — which is the motivation for re-optimizing the prompt for the target model.

## Section 6 — Hill Climbing on the Target Model

Each generation: the optimizer proposes `K` candidate prompts (informed by the incumbent's
judge reasoning), each is fanned out across Runtime sessions against the target and scored,
the best becomes the new incumbent, and we stop when the accuracy gain falls below threshold.

In [ ]:
from hillclimb import run_hill_climb
from committed_run_loader import load_committed_hill_climb

if mc.RUN.use_committed_run:
    result = load_committed_hill_climb()
else:
    # LIVE: wire the optimizer (fixed model) and the fan-out+score function.
    from optimizer import make_mutate_fn
    # complete(prompt)->str against mc.OPTIMIZER_MODEL via the Gateway; and
    # fanout_score(prompt)->CandidateScore deploying K*sessions on Runtime and scoring
    # with the AgentCore evaluators. Both are built from the helpers in Sections 2-4.
    raise NotImplementedError('Set use_committed_run=False only after wiring the live '
                              'fan-out/score + optimizer helpers (scripts/run_optimization.py).')

print('baseline accuracy :', result.baseline.accuracy)
for g in result.generations:
    bests = ', '.join(f'{c.accuracy:.2f}' for c in g.candidates)
    print(f'gen {g.index}: candidates [{bests}] -> best {g.best.accuracy:.2f}')
print('stopped:', result.stopped_reason)
print('best accuracy:', result.best.accuracy, '(+%.2f over baseline)' % result.accuracy_gain)

In [ ]:
import matplotlib.pyplot as plt

xs = [0] + [g.index for g in result.generations]
incumbent = [result.baseline.accuracy]
for g in result.generations:
    incumbent.append(max(incumbent[-1], g.best.accuracy))
fig, ax = plt.subplots(figsize=(7,4))
for g in result.generations:
    ax.scatter([g.index]*len(g.candidates), [c.accuracy for c in g.candidates],
               color='#9aa', label='candidates' if g.index==1 else None, zorder=2)
ax.plot(xs, incumbent, color='#2a6', marker='o', linewidth=2, label='incumbent (best so far)', zorder=3)
ax.axhline(result.baseline.accuracy, color='#c55', linestyle='--', label='baseline (unoptimized target)')
ax.set_xlabel('generation'); ax.set_ylabel('goal-completion accuracy')
ax.set_title('Hill climbing the prompt on the target model'); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## Section 7 — Why the Score Moved

The value of keeping the loop in the notebook: read the judge's **reasoning** on the winning
and losing candidates, so each prompt change is explainable, not magic.

In [ ]:
for g in result.generations:
    print(f'=== generation {g.index} ===')
    best = g.best
    for c in g.candidates:
        tag = ' (WINNER)' if c is best else ''
        print(f'  acc {c.accuracy:.2f}{tag}: {c.prompt[:80]}')
        for r in c.judge_reasoning:
            print(f'      - {r}')
    print()

## Section 8 — The Recommendation

The deliverable: source baseline vs target baseline vs **target optimized**, on all three axes,
and the winning prompt.

In [ ]:
winner = result.best
rows = [
    {'stage': 'source baseline (GPT-5.6 Luna)', 'accuracy': src.accuracy,
     'latency_p50_s': src.latency_p50_s, 'cost_usd_per_session': src.cost_usd_per_session},
    {'stage': 'target baseline (GPT-6 Luna, same prompt)', 'accuracy': tgt.accuracy,
     'latency_p50_s': tgt.latency_p50_s, 'cost_usd_per_session': tgt.cost_usd_per_session},
    {'stage': 'target OPTIMIZED (hill climb)', 'accuracy': winner.accuracy,
     'latency_p50_s': winner.latency_p50_s, 'cost_usd_per_session': winner.cost_usd_per_session},
]
import pandas as pd; display(pd.DataFrame(rows))
print('\nRecommended prompt for the target model:\n')
print(winner.prompt)

**How to read this.** We ran the loop until accuracy stopped improving meaningfully. The
recommended prompt scores highest on the target model, at the same-or-better latency and cost
as the source baseline.

**Honest caveats.** The judge is itself a model — calibrate it against human labels before
trusting verdicts (see the Chatbot module's judge-validation step). Gains are measured on this
gold set; re-run on drift and on new model launches. Latency and cost are whole-conversation
(multi-turn) figures, not single calls.

## Section 9 — Turning It Up & Cleanup

To run live on your own data: point the config at `mc.PRODUCTION_RUN_EXAMPLE` (more candidates,
sessions, scenarios, generations) and set `use_committed_run=False`, then either run this
notebook or `python scripts/run_optimization.py all`. Tear down with the Gateway/Runtime
cleanup helpers when done.

In [ ]:
print('Scale knobs (edit model_config.py):')
for f in ('candidates_per_generation','sessions_per_candidate','scenarios_per_session',
          'max_generations','min_accuracy_gain_to_continue'):
    print(f'  RUN.{f} =', getattr(mc.RUN, f), '| PRODUCTION_RUN_EXAMPLE.' + f, '=',
          getattr(mc.PRODUCTION_RUN_EXAMPLE, f))
# Cleanup (live only):
# !python gateway/setup_gateway.py teardown